In [1]:
from __future__ import annotations

import json
import logging
import math
import time
from collections import deque
from dataclasses import dataclass
from pathlib import Path
from typing import List, Literal, Optional, Tuple

import gymnasium as gym
import numpy as np
from tqdm import tqdm
import os
import cv2
import imageio


In [2]:


# ═════════════════════════════════════════════════════════════════════════════
# 1.  Binning  – state discretisation
# ═════════════════════════════════════════════════════════════════════════════


class Binning:

    def __init__(self):
        angles = [0, 45, 90, 120, 150, 180, 210, 240, 270, 315, 360, 0]

        theta1Av_Limit, theta1Av_inc, theta1Av_base = math.pi * 4, 1, 3
        theta2Av_Limit, theta2Av_inc, theta2Av_base = math.pi * 9, 2, 4

        self.theta_cos_div = [np.cos(math.radians(a)) for a in angles]
        self.theta_sin_div = [np.sin(math.radians(a)) for a in angles]
        self.theta1Av = self.velocity_binG(theta1Av_base, theta1Av_inc, theta1Av_Limit)
        self.theta2Av = self.velocity_binG(theta2Av_base, theta2Av_inc, theta2Av_Limit)

        self.bin_map = {"theta1": self.theta1Av, "theta2": self.theta2Av}

        # Number of bins per dimension (edges - 1 = number of intervals)
        self.n_angle_bins = len(self.theta_cos_div) - 1   # 11
        self.n_theta1_bins = len(self.theta1Av) - 1
        self.n_theta2_bins = len(self.theta2Av) - 1

    def velocity_binG(self, base: float, inc: float, limit: float) -> List[float]:
        bins = []
        present = base

        while present < limit:
            bins.append(present)
            bins.append(-present)
            present += base
            base += inc

        bins.extend([limit, -limit])
        bins.sort()
        return bins

    def _bin_finder(self, bin_left: float, bin_right: float, key: float) -> bool:
        if bin_left <= bin_right:
            return bin_left <= key < bin_right
        else:
            return bin_right < key <= bin_left

    def _binSearch(self, search_list: List[float], key: float) -> List[int]:
        return [
            i
            for i in range(len(search_list) - 1)
            if self._bin_finder(search_list[i], search_list[i + 1], key)
        ]

    def pos(self, pos: tuple[float, float]) -> int:
        cos_val, sin_val = pos

        valid_cos_bins = self._binSearch(self.theta_cos_div, cos_val)
        valid_sin_bins = self._binSearch(self.theta_sin_div, sin_val)

        common = list(set(valid_cos_bins) & set(valid_sin_bins))

        # BUG FIX: a list is never None; use `not common` to detect empty intersection.
        # When empty, fall back to the union average; if that too is empty return 0.
        if not common:
            union = set(valid_cos_bins) | set(valid_sin_bins)
            if not union:
                return 0
            return math.ceil(math.fsum(union) / len(union))

        return common[0]

    def angular(self, angularVelocity: float, bin_search: Literal["theta1", "theta2"]) -> int:
        bin_list = self.bin_map.get(bin_search)
        bins = self._binSearch(search_list=bin_list, key=angularVelocity)
        if bins:
            return bins[0]
        # BUG FIX: velocity outside bin edges (e.g. exactly ±limit).
        # Clamp to the nearest edge bin so the return is always a valid int index.
        # Returning None would silently corrupt numpy Q-table indexing (None → np.newaxis).
        return 0 if angularVelocity <= bin_list[0] else len(bin_list) - 2


In [3]:


# ═════════════════════════════════════════════════════════════════════════════
# 2.  Policy  – Q-table with epsilon-greedy action selection
# ═════════════════════════════════════════════════════════════════════════════


class Policy:

    def __init__(self, action_space: int, learning_rate: float = 0.1, discount_factor: float = 0.99):
        self.action_space = action_space
        self.binner = Binning()
        self.learning_rate = learning_rate
        self.discount_factor = discount_factor

        # Pre-allocate a fixed-size Q-table — replaces the unbounded dict.
        # Shape: (θ1_pos_bins, θ2_pos_bins, θ1_vel_bins, θ2_vel_bins, actions)
        # Size is determined entirely by Binning and never grows during training.
        self.q_table = np.zeros(
            (
                self.binner.n_angle_bins,
                self.binner.n_angle_bins,
                self.binner.n_theta1_bins,
                self.binner.n_theta2_bins,
                action_space,
            ),
            dtype=np.float32,
        )

    def discretize_state(self, state: Tuple[float, float, float, float, float, float]) -> Tuple[int, int, int, int]:
        theta1_pos_bin = self.binner.pos((state[0], state[1]))
        theta2_pos_bin = self.binner.pos((state[2], state[3]))
        theta1_vel_bin = self.binner.angular(state[4], "theta1")
        theta2_vel_bin = self.binner.angular(state[5], "theta2")
        return (theta1_pos_bin, theta2_pos_bin, theta1_vel_bin, theta2_vel_bin)

    def update(
        self,
        state: Tuple[int, int, int, int],
        action: int,
        reward: float,
        next_state: Tuple[int, int, int, int],
    ) -> float:
        best_next_q = float(self.q_table[next_state].max())
        td_target = reward + self.discount_factor * best_next_q
        td_error = td_target - float(self.q_table[state][action])
        self.q_table[state][action] += self.learning_rate * td_error
        return td_error

    def epsilon_greedy(self, state: Tuple[int, int, int, int], epsilon: float) -> int:
        if np.random.rand() < epsilon:
            return np.random.randint(self.action_space)
        return int(np.argmax(self.q_table[state]))

    def act(self, observation: Tuple) -> int:
        state = self.discretize_state(observation)
        return int(np.argmax(self.q_table[state]))

    def q_table_array(self) -> np.ndarray:
        return self.q_table

    def mean_entropy(self) -> float:
        flat = self.q_table.reshape(-1, self.action_space).astype(np.float64)
        q_shifted = flat - flat.max(axis=1, keepdims=True)
        exp_q = np.exp(q_shifted)
        probs = exp_q / exp_q.sum(axis=1, keepdims=True)
        probs = np.clip(probs, 1e-12, None)
        return float(-np.sum(probs * np.log(probs), axis=1).mean())



In [4]:

# ═════════════════════════════════════════════════════════════════════════════
# 3.  Logger setup
# ═════════════════════════════════════════════════════════════════════════════


def build_logger(name: str = "ql_agent", log_dir: str = "logs") -> logging.Logger:
    """
    Two handlers:
      • StreamHandler  – WARNING and above to console (don't clutter tqdm)
      • FileHandler    – DEBUG and above to  logs/ql_agent.log
    """
    Path(log_dir).mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger(name)
    logger.setLevel(logging.DEBUG)

    if not logger.handlers:
        fmt = logging.Formatter(
            "%(asctime)s | %(levelname)-8s | %(message)s",
            datefmt="%H:%M:%S",
        )
        fh = logging.FileHandler(f"{log_dir}/{name}.log", mode="w")
        fh.setLevel(logging.DEBUG)
        fh.setFormatter(fmt)

        ch = logging.StreamHandler()
        ch.setLevel(logging.WARNING)
        ch.setFormatter(fmt)

        logger.addHandler(fh)
        logger.addHandler(ch)

    return logger



In [5]:

# ═════════════════════════════════════════════════════════════════════════════
# 4.  DebugSnapshot  – frozen Q-table diagnostic at one episode
# ═════════════════════════════════════════════════════════════════════════════


@dataclass
class DebugSnapshot:
    episode: int
    q_min: float
    q_max: float
    q_mean: float
    q_std: float
    dead_states: int        # rows where all Q-values == 0
    action_dist: list[int]  # count of argmax per action index
    convergence_delta: float

    def report(self) -> str:
        lines = [
            f"── debug snapshot  ep={self.episode} ──────────────",
            f"  Q range          [{self.q_min:+.4f}, {self.q_max:+.4f}]",
            f"  Q mean / std     {self.q_mean:+.4f} / {self.q_std:.4f}",
            f"  dead states      {self.dead_states}  (all-zero Q rows)",
            f"  convergence Δ    {self.convergence_delta:.6f}",
            f"  dominant action  {self.action_dist}  (argmax histogram)",
            "─" * 50,
        ]
        return "\n".join(lines)



In [6]:

# ═════════════════════════════════════════════════════════════════════════════
# 5.  TrainingTracer  – lightweight metric collector
# ═════════════════════════════════════════════════════════════════════════════


class TrainingTracer:
    """
    Attach to QLearningAgent.  Call:
      • log_step(td_error)                                  inside the step loop
      • log_episode(reward, length, epsilon, policy)        at episode end
      • snapshot(q_table, episode)                          every N episodes
    """

    def __init__(
        self,
        window: int = 50,
        snapshot_every: int = 100,
        logger: Optional[logging.Logger] = None,
    ):
        self.window = window
        self.snapshot_every = snapshot_every
        self.logger = logger or build_logger()

        self.episode_rewards: list[float] = []
        self.episode_lengths: list[int] = []
        self.epsilon_history: list[float] = []
        self.q_max_history: list[float] = []
        self.q_mean_history: list[float] = []
        self.policy_entropy: list[float] = []

        self.td_errors: deque[float] = deque(maxlen=100_000)
        self.snapshots: list[DebugSnapshot] = []

        self._prev_q: Optional[np.ndarray] = None
        self._ep_td_buf: list[float] = []
        self._wall_start: float = time.perf_counter()

    # ── step-level hook ───────────────────────────────────────────────────────

    def log_step(self, td_error: float) -> None:
        self.td_errors.append(td_error)
        self._ep_td_buf.append(td_error)

    # ── episode-level hook ────────────────────────────────────────────────────

    def log_episode(
        self,
        episode: int,
        reward: float,
        length: int,
        epsilon: float,
        q_table: np.ndarray,
        policy,
    ) -> dict:
        self.episode_rewards.append(reward)
        self.episode_lengths.append(length)
        self.epsilon_history.append(epsilon)
        self.q_max_history.append(float(q_table.max()))
        self.q_mean_history.append(float(q_table.mean()))
        self.policy_entropy.append(float(policy.mean_entropy()))

        recent = self.episode_rewards[-self.window:]
        avg_r = float(np.mean(recent))
        avg_td = float(np.mean(self._ep_td_buf)) if self._ep_td_buf else 0.0
        self._ep_td_buf.clear()

        elapsed = time.perf_counter() - self._wall_start

        self.logger.debug(
            "ep=%d  reward=%.2f  avg%d=%.2f  len=%d  ε=%.4f  "
            "q_max=%.4f  q_mean=%.4f  td_err=%.4f  entropy=%.4f  t=%.1fs",
            episode, reward, self.window, avg_r, length, epsilon,
            q_table.max(), q_table.mean(), avg_td, self.policy_entropy[-1], elapsed,
        )

        if abs(avg_td) > 10:
            self.logger.warning("ep=%d  large TD error avg=%.4f — check reward scale or α", episode, avg_td)
        if q_table.max() > 1e4:
            self.logger.warning("ep=%d  Q-value explosion: max=%.2f", episode, q_table.max())

        return {
            "r": f"{reward:.1f}",
            f"avg{self.window}": f"{avg_r:.1f}",
            "ε": f"{epsilon:.3f}",
            "td": f"{avg_td:.4f}",
            "H": f"{self.policy_entropy[-1]:.3f}",
        }

    # ── deep diagnostic snapshot ──────────────────────────────────────────────

    def snapshot(self, q_table: np.ndarray, episode: int) -> DebugSnapshot:
        flat = q_table.reshape(-1, q_table.shape[-1])
        dead = int((flat == 0).all(axis=1).sum())
        argmaxes = np.argmax(flat, axis=1)
        action_dist = [int((argmaxes == a).sum()) for a in range(q_table.shape[-1])]

        delta = float(np.abs(q_table - self._prev_q).mean()) if self._prev_q is not None else float("nan")
        self._prev_q = q_table.copy()

        snap = DebugSnapshot(
            episode=episode,
            q_min=float(q_table.min()),
            q_max=float(q_table.max()),
            q_mean=float(q_table.mean()),
            q_std=float(q_table.std()),
            dead_states=dead,
            action_dist=action_dist,
            convergence_delta=delta,
        )
        self.snapshots.append(snap)
        self.logger.info("\n%s", snap.report())
        return snap

    # ── convergence check ─────────────────────────────────────────────────────

    def has_converged(self, threshold: float = 1e-4, window: int = 5) -> bool:
        recent = [
            s.convergence_delta
            for s in self.snapshots[-window:]
            if not np.isnan(s.convergence_delta)
        ]
        return len(recent) >= window and max(recent) < threshold

    # ── serialise / plot ──────────────────────────────────────────────────────

    def save(self, path: str = "trace.json") -> None:
        data = {
            "episode_rewards": self.episode_rewards,
            "episode_lengths": self.episode_lengths,
            "epsilon_history": self.epsilon_history,
            "q_max_history": self.q_max_history,
            "q_mean_history": self.q_mean_history,
            "policy_entropy": self.policy_entropy,
            "td_errors": list(self.td_errors),
            "snapshots": [{k: v for k, v in vars(s).items()} for s in self.snapshots],
        }
        with open(path, "w") as f:
            json.dump(data, f, indent=2)
        self.logger.info("Trace saved → %s", path)

    def plot_training(self, save_dir: str = "plots") -> dict:
        import matplotlib.pyplot as plt
        import numpy as np
        from pathlib import Path

        BG        = "#000000"
        MUTED_W   = "#5a5a6e"
        RAW_R     = "#7dae9b"
        ROLL_R    = "#ffd43b"
        TD_CLR    = "#ff6b6b"
        QMAX_CLR  = "#c77dff"
        QMEAN_CLR = "#38d9a9"
        ENT_CLR   = "#ff922b"
        EPS_CLR   = "#74c0fc"

        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)

        eps  = range(1, len(self.episode_rewards) + 1)
        roll = lambda x: np.convolve(x, np.ones(self.window) / self.window, mode="valid")

        def style_ax(ax, ylabel_color="#e8e8f0"):
            ax.set_facecolor(BG)
            ax.tick_params(colors=MUTED_W, labelsize=8)
            ax.xaxis.label.set_color(MUTED_W)
            ax.yaxis.label.set_color(ylabel_color)
            ax.title.set_color("#e8e8f0")
            for spine in ax.spines.values():
                spine.set_color(MUTED_W)
                spine.set_linewidth(0.5)
            ax.grid(True, color=MUTED_W, linewidth=0.3, alpha=0.4, linestyle="--")

        def legend(ax):
            leg = ax.legend(fontsize=8, facecolor="#1a1a2e",
                            edgecolor=MUTED_W, labelcolor="#e8e8f0",
                            framealpha=0.9)
            leg.get_frame().set_linewidth(0.5)

        saved_paths = {}

        # ── 1. Episode reward ─────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(6, 4))
        fig.patch.set_facecolor(BG)
        style_ax(ax)

        ax.plot(eps, self.episode_rewards,
                alpha=0.18, color=RAW_R, lw=0.7, label="raw")
        ax.plot(range(self.window, len(self.episode_rewards) + 1),
                roll(self.episode_rewards),
                color=ROLL_R, lw=1.8, label=f"rolling {self.window}")

        ax.set_title("Episode Reward")
        ax.set_xlabel("episode")
        legend(ax)

        path = save_dir / "episode_reward.png"
        fig.savefig(path, dpi=150, facecolor=fig.get_facecolor(), bbox_inches="tight")
        plt.close(fig)
        saved_paths["reward"] = str(path.resolve())

        # ── 2. TD error ───────────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(6, 4))
        fig.patch.set_facecolor(BG)
        style_ax(ax)

        ax.plot(list(self.td_errors), alpha=0.55, color=TD_CLR, lw=0.6)

        ax.set_title("TD Error (per step)")
        ax.set_xlabel("step")

        path = save_dir / "td_error.png"
        fig.savefig(path, dpi=150, facecolor=fig.get_facecolor(), bbox_inches="tight")
        plt.close(fig)
        saved_paths["td_error"] = str(path.resolve())

        # ── 3. Q-value evolution ──────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(6, 4))
        fig.patch.set_facecolor(BG)
        style_ax(ax)

        ax.plot(eps, self.q_max_history,  label="Q max",  color=QMAX_CLR, lw=1.4)
        ax.plot(eps, self.q_mean_history, label="Q mean",
                color=QMEAN_CLR, lw=1.0, linestyle="--")

        ax.set_title("Q-value Evolution")
        ax.set_xlabel("episode")
        legend(ax)

        path = save_dir / "q_values.png"
        fig.savefig(path, dpi=150, facecolor=fig.get_facecolor(), bbox_inches="tight")
        plt.close(fig)
        saved_paths["q_values"] = str(path.resolve())

        # ── 4. Exploration signals ────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(6, 4))
        fig.patch.set_facecolor(BG)
        style_ax(ax, ylabel_color=ENT_CLR)

        ax2 = ax.twinx()
        ax2.set_facecolor(BG)
        ax2.tick_params(colors=MUTED_W, labelsize=8)
        ax2.yaxis.label.set_color(EPS_CLR)

        for spine in ax2.spines.values():
            spine.set_color(MUTED_W)
            spine.set_linewidth(0.5)

        ax.plot(eps, self.policy_entropy,
                color=ENT_CLR, lw=1.4, label="entropy H")
        ax2.plot(eps, self.epsilon_history,
                color=EPS_CLR, lw=1.0, linestyle="--", label="ε")

        ax.set_title("Exploration Signals")
        ax.set_xlabel("episode")
        ax.set_ylabel("H (entropy)", color=ENT_CLR)
        ax2.set_ylabel("ε", color=EPS_CLR)

        lines  = ax.get_lines() + ax2.get_lines()
        labels = [l.get_label() for l in lines]
        ax.legend(lines, labels, fontsize=8,
                facecolor="#1a1a2e", edgecolor=MUTED_W,
                labelcolor="#e8e8f0", framealpha=0.9)

        path = save_dir / "exploration.png"
        fig.savefig(path, dpi=150, facecolor=fig.get_facecolor(), bbox_inches="tight")
        plt.close(fig)
        saved_paths["exploration"] = str(path.resolve())

        self.logger.info("Plots saved → %s", save_dir.resolve())
        return saved_paths

In [7]:

# ═════════════════════════════════════════════════════════════════════════════
# 6.  QLearningAgent  – training loop with full tracing
# ═════════════════════════════════════════════════════════════════════════════


class QLearningAgent:

    def __init__(
        self,
        action_space: int = 3,
        learning_rate: float = 0.1,
        discount_factor: float = 0.99,
        num_episodes: int = 1000,
        epsilon_start: float = 1.0,
        epsilon_end: float = 0.05,
        epsilon_decay: float = 0.995,
        snapshot_every: int = 100,
        early_stop_delta: float = 1e-4,
        log_dir: str = "logs",
    ):
        self.action_space = action_space
        self.learning_rate = learning_rate
        self.discount_factor = discount_factor
        self.num_episodes = num_episodes
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.early_stop_delta = early_stop_delta

        self.policy = Policy(
            action_space=action_space,
            learning_rate=learning_rate,
            discount_factor=discount_factor,
        )
        self.environment = gym.make("Acrobot-v1")#, render_mode="human")

        self.logger = build_logger("ql_agent", log_dir)
        self.tracer = TrainingTracer(
            window=500,
            snapshot_every=snapshot_every,
            logger=self.logger,
        )

        self.logger.info(
            "Agent init  episodes=%d  α=%.3f  γ=%.3f  ε∈[%.2f,%.2f]",
            num_episodes, learning_rate, discount_factor, epsilon_end, epsilon_start,
        )

    # ── training loop ─────────────────────────────────────────────────────────

    def training_step(self, avg_episode_length: int) -> None:
        pbar = tqdm(range(self.num_episodes), desc="training", unit="ep", dynamic_ncols=True)

        for episode in pbar:
            obs, info = self.environment.reset()
            ep_reward = 0.0
            ep_length = 0

            for step in range(avg_episode_length):
                dobs = self.policy.discretize_state(obs)
                action = self.policy.epsilon_greedy(dobs, self.epsilon)

                next_obs, reward, terminated, truncated, info = self.environment.step(action)
                ep_reward += reward
                ep_length += 1

                dnext_obs = self.policy.discretize_state(next_obs)
                td_error = self.policy.update(
                    state=dobs, action=action, reward=reward, next_state=dnext_obs,
                )
                self.tracer.log_step(td_error)

                obs = next_obs
                if terminated or truncated:
                    break

            self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)

            postfix = self.tracer.log_episode(
                episode=episode,
                reward=ep_reward,
                length=ep_length,
                epsilon=self.epsilon,
                q_table=self.policy.q_table,
                policy=self.policy,
            )
            pbar.set_postfix(postfix)

            if (episode + 1) % self.tracer.snapshot_every == 0:
                self.tracer.snapshot(self.policy.q_table, episode + 1)

            if self.tracer.has_converged(threshold=self.early_stop_delta):
                self.logger.warning(
                    "Early stop at ep=%d  (convergence Δ < %.1e)", episode + 1, self.early_stop_delta,
                )
                tqdm.write(f"[early stop] converged at episode {episode + 1}")
                break

        self.environment.close()
        self._summarise()

    # ── post-training ─────────────────────────────────────────────────────────

    def _summarise(self) -> None:
        rewards = self.tracer.episode_rewards
        self.logger.info(
            "Training done  episodes=%d  best=%.1f  worst=%.1f  final_avg50=%.1f",
            len(rewards), max(rewards), min(rewards), float(np.mean(rewards[-50:])),
        )
        tqdm.write(
            f"\nDone  |  episodes={len(rewards)}"
            f"  best={max(rewards):.1f}"
            f"  avg50={np.mean(rewards[-50:]):.1f}"
            f"  final ε={self.epsilon:.4f}"
        )

    def debug_report(self) -> str:
        if not self.tracer.snapshots:
            return "No snapshots yet. Run training_step() first."
        return "\n".join(s.report() for s in self.tracer.snapshots)

    def save_trace(self, path: str = "trace.json") -> None:
        self.tracer.save(path)

    def plot(self, save_path: Optional[str] = None) -> None:
        self.tracer.plot_training(save_path)

    # ── model save / load ─────────────────────────────────────────────────────

    def save_model(self, path: str = "acrobot_model.npz") -> None:
        np.savez_compressed(
            path,
            q_table=self.policy.q_table,
            epsilon=np.array([self.epsilon]),
            action_space=np.array([self.action_space]),
            learning_rate=np.array([self.learning_rate]),
            discount_factor=np.array([self.discount_factor]),
        )
        self.logger.info("Model saved → %s", path)
        print(f"Model saved → {path}")

    @classmethod
    def load_model(cls, path: str = "acrobot_model.npz") -> "QLearningAgent":
        data = np.load(path)
        action_space = int(data["action_space"][0])
        agent = cls(
            action_space=action_space,
            learning_rate=float(data["learning_rate"][0]),
            discount_factor=float(data["discount_factor"][0]),
            epsilon_start=float(data["epsilon"][0]),
        )
        agent.epsilon = float(data["epsilon"][0])
        agent.policy.q_table = data["q_table"]
        agent.logger.info("Model loaded ← %s", path)
        print(f"Model loaded ← {path}  (ε={agent.epsilon:.4f})")
        return agent

    # ── inference ─────────────────────────────────────────────────────────────

    def run_inference_view(
        self,
        num_episodes: int = 5,
        max_steps: int = 500,
        render: bool = True,
    ) -> list[dict]:
        env = gym.make("Acrobot-v1", render_mode="human" if render else None)
        results = []

        for ep in range(num_episodes):
            obs, info = env.reset()
            ep_reward = 0.0

            for step in range(max_steps):
                action = self.policy.act(obs)
                obs, reward, terminated, truncated, info = env.step(action)
                ep_reward += reward
                if terminated or truncated:
                    break

            results.append({"episode": ep + 1, "reward": ep_reward, "steps": step + 1})
            print(f"  Episode {ep + 1:>3d}:  reward={ep_reward:.1f}  steps={step + 1}")

        env.close()

        avg_reward = np.mean([r["reward"] for r in results])
        avg_steps = np.mean([r["steps"] for r in results])
        print(f"\nInference summary ({num_episodes} episodes):")
        print(f"  avg reward = {avg_reward:.1f}   avg steps = {avg_steps:.1f}")
        return results

    def run_inference(
        self,
        num_episodes: int = 5,
        max_steps: int = 500,
        video_folder: str = "videos",
    ):
        os.makedirs(video_folder, exist_ok=True)
        env = gym.make("Acrobot-v1", render_mode="rgb_array")
        results = []

        for ep in range(num_episodes):
            obs, _ = env.reset()
            ep_reward = 0.0
            trajectory = []
            action_counts = {}
            frames = []

            for step in range(max_steps):
                # Gym renders RGB → convert to BGR for OpenCV
                frame = cv2.cvtColor(env.render(), cv2.COLOR_RGB2BGR)

                action = self.policy.act(obs)
                if action is None:
                    action = env.action_space.sample()

                action_counts[action] = action_counts.get(action, 0) + 1

                # Draw HUD overlay
                cv2.putText(
                    frame,
                    f"Ep:{ep + 1}  Step:{step}  Reward:{ep_reward:.1f}",
                    (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (255, 255, 255),
                    2,
                    cv2.LINE_AA,
                )

                for i, (act, count) in enumerate(action_counts.items()):
                    cv2.putText(
                        frame,
                        f"Action {act}: {count}",
                        (10, 50 + i * 20),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.5,
                        (0, 255, 0),
                        1,
                        cv2.LINE_AA,
                    )

                frames.append(frame)

                next_obs, reward, terminated, truncated, _ = env.step(action)
                trajectory.append({"obs": obs, "action": action, "reward": reward})
                obs = next_obs
                ep_reward += reward

                if terminated or truncated:
                    # Capture the final post-step frame before breaking
                    final_frame = cv2.cvtColor(env.render(), cv2.COLOR_RGB2BGR)
                    cv2.putText(
                        final_frame,
                        f"Ep:{ep + 1}  Step:{step + 1}  Reward:{ep_reward:.1f}",
                        (10, 25),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (255, 255, 255),
                        2,
                        cv2.LINE_AA,
                    )
                    for i, (act, count) in enumerate(action_counts.items()):
                        cv2.putText(
                            final_frame,
                            f"Action {act}: {count}",
                            (10, 50 + i * 20),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.5,
                            (0, 255, 0),
                            1,
                            cv2.LINE_AA,
                        )
                    frames.append(final_frame)
                    break

            # Save episode video
            height, width = frames[0].shape[:2]
            video_path = os.path.join(video_folder, f"episode_{ep + 1}.mp4")

            writer = cv2.VideoWriter(
                video_path,
                cv2.VideoWriter_fourcc(*"mp4v"),
                30,
                (width, height),
            )
            for frame in frames:
                writer.write(frame)
            writer.release()

            results.append({
                "episode":       ep + 1,
                "reward":        ep_reward,
                "steps":         step + 1,
                "trajectory":    trajectory,
                "action_counts": action_counts,
            })
            print(f"Episode {ep + 1}: reward={ep_reward:.1f}  steps={step + 1}  → {video_path}")

        env.close()
        return results

In [ ]:


# ═════════════════════════════════════════════════════════════════════════════
# 7.  Training
# ═════════════════════════════════════════════════════════════════════════════

agent = QLearningAgent(
    num_episodes=50_000,
    snapshot_every=500,
    early_stop_delta=-1e-4,
)

agent.training_step(avg_episode_length=5_00)

print(agent.debug_report())
agent.save_trace("trace.json")
agent.save_model("acrobot_model.npz")
agent.plot("training.png")


In [ ]:

# ═════════════════════════════════════════════════════════════════════════════
# 8.  Inference  – 1 million episodes (headless, no video)
# ═════════════════════════════════════════════════════════════════════════════

NUM_EPISODES = 1_000_000
MAX_STEPS = 500

# Load trained model (or use the agent already in memory)
try:
    agent
except NameError:
    agent = QLearningAgent.load_model("acrobot_model.npz")

env = gym.make("Acrobot-v1")

rewards = np.empty(NUM_EPISODES, dtype=np.float64)
steps = np.empty(NUM_EPISODES, dtype=np.int32)

pbar = tqdm(range(NUM_EPISODES), desc="inference", unit="ep", dynamic_ncols=True)
for ep in pbar:
    obs, _ = env.reset()
    ep_reward = 0.0

    for step in range(MAX_STEPS):
        action = agent.policy.act(obs)
        obs, reward, terminated, truncated, _ = env.step(action)
        ep_reward += reward
        if terminated or truncated:
            break

    rewards[ep] = ep_reward
    steps[ep] = step + 1

    if (ep + 1) % 100_000 == 0:
        pbar.set_postfix(
            avg_r=f"{rewards[:ep+1].mean():.1f}",
            avg_s=f"{steps[:ep+1].mean():.1f}",
        )

env.close()

# ── Summary stats ─────────────────────────────────────────────────────────
print(f"\n{'═' * 60}")
print(f"Inference complete: {NUM_EPISODES:,} episodes")
print(f"{'═' * 60}")
print(f"  Mean reward  : {rewards.mean():.2f}  ± {rewards.std():.2f}")
print(f"  Median reward: {np.median(rewards):.1f}")
print(f"  Best reward  : {rewards.max():.1f}")
print(f"  Worst reward : {rewards.min():.1f}")
print(f"  Mean steps   : {steps.mean():.1f}  ± {steps.std():.1f}")
print(f"  Solved (≥-100): {(rewards >= -100).sum():,} / {NUM_EPISODES:,}"
      f"  ({(rewards >= -100).mean() * 100:.2f}%)")

# Save results
inference_results = {
    "num_episodes": NUM_EPISODES,
    "mean_reward": float(rewards.mean()),
    "std_reward": float(rewards.std()),
    "median_reward": float(np.median(rewards)),
    "best_reward": float(rewards.max()),
    "worst_reward": float(rewards.min()),
    "mean_steps": float(steps.mean()),
    "std_steps": float(steps.std()),
    "solve_rate": float((rewards >= -100).mean()),
}
with open("inference_results.json", "w") as f:
    json.dump(inference_results, f, indent=2)
print(f"\nResults saved → inference_results.json")


In [ ]:

# ═════════════════════════════════════════════════════════════════════════════
# 9.  Histogram  – steps distribution across 1M inference episodes
# ═════════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt

plt.style.use("dark_background")
fig, ax = plt.subplots(figsize=(10, 6))

counts, bin_edges, patches = ax.hist(
    steps, bins=50, color="#00bcd4", edgecolor="white", linewidth=0.5
)

# Highlight the bin containing the max-step (500) cutoff
for patch, left, right in zip(patches, bin_edges[:-1], bin_edges[1:]):
    if left <= MAX_STEPS <= right:
        patch.set_facecolor("#ff4444")
        patch.set_edgecolor("yellow")
        patch.set_linewidth(2)
        idx = np.where(bin_edges[:-1] == left)[0][0]
        ax.annotate(
            f"{MAX_STEPS}\n(count: {int(counts[idx])})",
            xy=(MAX_STEPS, counts[idx]),
            xytext=(MAX_STEPS + 20, counts[idx] + max(counts) * 0.08),
            arrowprops=dict(arrowstyle="->", color="yellow", lw=1.5),
            fontsize=11, color="yellow", fontweight="bold",
        )

ax.set_title(f"Steps per Episode  ({NUM_EPISODES:,} episodes)", fontsize=16)
ax.set_xlabel("Steps", fontsize=13)
ax.set_ylabel("Frequency", fontsize=13)

# Add summary text box
textstr = (
    f"Mean: {steps.mean():.1f}\n"
    f"Median: {np.median(steps):.0f}\n"
    f"Solve rate: {(rewards >= -100).mean() * 100:.2f}%"
)
ax.text(
    0.98, 0.95, textstr, transform=ax.transAxes,
    fontsize=11, verticalalignment="top", horizontalalignment="right",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="#1a1a2e", edgecolor="#5a5a6e", alpha=0.9),
    color="#e8e8f0",
)

fig.tight_layout()
plt.savefig("steps_histogram.png", dpi=150, facecolor="black")
plt.show()
print(f"Histogram saved → steps_histogram.png")
